<a href="https://colab.research.google.com/github/farrelrassya/Transformers-in-action/blob/main/01.Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1: Foundations of Modern Transformer Models

The transformer architecture arrived at **NIPS 2017** via the now-legendary paper *"Attention Is All You Need"* (Vaswani et al.), and it fundamentally reshaped how we build models for language -- and eventually for vision, audio, protein folding, and far beyond. To appreciate why this matters, we need to understand what transformers replaced and why.

### Why Transformers Changed Everything

Before transformers, the dominant architectures for sequence modeling were **Recurrent Neural Networks (RNNs)** and their gated variants -- **LSTMs** and **GRUs**. These models process tokens one at a time, left to right, maintaining a hidden state $\mathbf{h}_t$ that summarizes everything seen so far:

$$\mathbf{h}_t = f(\mathbf{h}_{t-1}, \mathbf{x}_t)$$

This sequential bottleneck creates two serious problems. First, **long-range dependencies decay** -- by the time the model reaches token $t = 500$, information from token $t = 1$ has been compressed through hundreds of nonlinear transformations, making it unreliable. Second, **training cannot be parallelized** across time steps, because computing $\mathbf{h}_t$ requires $\mathbf{h}_{t-1}$. On modern GPUs designed for massive parallelism, this is an enormous waste of hardware.

The transformer's key insight is the **self-attention mechanism**, which lets every token attend to every other token *simultaneously* -- no sequential processing required. For a sequence of length $n$, self-attention computes all pairwise relationships in a single matrix operation, achieving $O(1)$ sequential depth (compared to $O(n)$ for RNNs) at the cost of $O(n^2)$ memory. This trade-off is overwhelmingly favorable for modern hardware, which is why transformers train orders of magnitude faster on the same data.

### The Garden Metaphor -- and Why It Matters Professionally

The textbook uses a garden analogy: prepare the soil before planting exotic flowers. From a practitioner's standpoint, this is not just pedagogical decoration -- it reflects a real pattern in industry. Engineers who skip foundational understanding and jump straight to fine-tuning GPT-style models frequently hit walls they cannot debug: attention heads producing uniform distributions, positional encodings causing length generalization failures, or training instabilities from improper layer normalization placement. **Understanding the "why" behind each architectural decision is what separates someone who uses transformers from someone who can fix, adapt, and improve them.**

### What Part 1 Will Cover

This foundational section will build up the transformer piece by piece -- the mathematical machinery of attention, how positional information is injected, how training objectives shape model behavior, and why specific design choices (residual connections, layer normalization, multi-head attention) are not arbitrary but solve concrete problems. Every component we study here reappears in the large-scale models of Parts 2 and 3 (GPT, BERT, T5, and their descendants), so the investment in depth now pays compound returns later.

## 1.1 The Transformers Breakthrough

The transformer architecture, introduced by Vaswani et al. in *"Attention Is All You Need"* (2017), solved a fundamental limitation that had plagued sequence models for years: the inability to efficiently capture relationships between distant elements in a sequence. The core innovation is the **attention mechanism** -- a component that lets every element in a sequence directly "look at" every other element, bypassing the sequential bottleneck entirely.

What makes this so powerful is generality. Although transformers were originally designed for NLP tasks like translation, the attention mechanism operates on *any* ordered collection of vectors. This is why transformers now dominate not just text, but also audio (Whisper), images (Vision Transformer), video (VideoMAE), and even protein structure prediction (AlphaFold 2). The architecture does not care what the tokens represent -- it only cares about the relationships between them.




### 1.1.1 Translation Before Transformers: The Encoder-Decoder RNN

To understand why transformers were necessary, we first need to see what they replaced. Consider translating "I don't speak French" into "Je ne parle pas français" using an RNN-based **encoder-decoder** architecture.

<img src="https://raw.githubusercontent.com/farrelrassya/Transformers-in-action/main/folder-picture/01.%20chapter%2001/figure%201.1.png" width="700">

The process works in two stages:

**The Encoder** reads the source sentence one token at a time, updating a hidden state $\mathbf{h}_t$ at each step:

$$\mathbf{h}_t = f(\mathbf{h}_{t-1}, \mathbf{x}_t)$$

where $\mathbf{x}_t$ is the embedding of the $t$-th input word and $f$ is a nonlinear function (typically an LSTM or GRU cell). After processing all four input words, the encoder produces a single **context vector** $\mathbf{c} = \mathbf{h}_T$ -- a fixed-length vector that must compress the entire meaning of "I don't speak French."

**The Decoder** then generates the French output one word at a time, conditioned on this context vector:

$$\mathbf{s}_t = g(\mathbf{s}_{t-1}, y_{t-1}, \mathbf{c})$$

where $\mathbf{s}_t$ is the decoder's hidden state, $y_{t-1}$ is the previously generated word, and $\mathbf{c}$ is the context vector from the encoder. At each step, a probability distribution over the target vocabulary is produced, and the most likely word is selected.

### The Critical Bottleneck

The fundamental problem is visible in the equations above: **everything the decoder knows about the source sentence must pass through the single vector $\mathbf{c}$**. For our short example sentence, this is manageable. But consider translating a paragraph of 200 words -- the encoder must squeeze 200 words of meaning into the same fixed-dimensional vector (typically $\mathbf{c} \in \mathbb{R}^{512}$ or $\mathbb{R}^{1024}$). Information inevitably gets lost.

This is known as the **information bottleneck problem**, and it manifests concretely: RNN-based translation quality degrades sharply as sentence length increases. Empirically, performance drops significantly beyond roughly 20-30 tokens.

There is a second, equally serious problem: **sequential computation**. Because $\mathbf{h}_t$ depends on $\mathbf{h}_{t-1}$, the encoder cannot process word 3 until word 2 is finished, which in turn waits for word 1. For a sequence of length $n$, this creates $O(n)$ sequential steps that **cannot be parallelized**. On a modern GPU with thousands of cores sitting idle waiting for each step to finish, this is an enormous waste of compute.

### From a Production Standpoint

These two problems -- information loss and sequential processing -- are not just theoretical concerns. In practice, they meant that pre-transformer translation systems required either (a) limiting input length, which frustrated users, or (b) training on very large hidden dimensions to compensate, which multiplied training time and cost. The attention mechanism, which we will examine next, solves both problems simultaneously: it removes the fixed bottleneck by letting the decoder access *all* encoder states, and the transformer architecture removes sequential dependence entirely by computing attention over the full sequence in parallel.

### 1.1.3 Unveiling the Attention Mechanism

The attention mechanism answers a simple but powerful question: **when the decoder generates a target word, which source words should it focus on?** Instead of compressing the entire input into a single fixed-length context vector $\mathbf{c}$ (as vanilla RNNs do), attention lets the decoder look back at *all* encoder hidden states and decide, dynamically, which ones matter most for the current output step.

Formally, at each decoder time step $t$, attention computes a weighted sum over all encoder hidden states $\mathbf{h}_1, \mathbf{h}_2, \ldots, \mathbf{h}_n$:

$$\mathbf{c}_t = \sum_{i=1}^{n} \alpha_{t,i} \, \mathbf{h}_i$$

where $\alpha_{t,i}$ is the **attention weight** -- a scalar between 0 and 1 indicating how relevant source word $i$ is to generating the current target word. These weights are computed via a softmax over alignment scores:

$$\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_{j=1}^{n} \exp(e_{t,j})}, \qquad e_{t,i} = \text{score}(\mathbf{s}_{t-1}, \mathbf{h}_i)$$

where $\mathbf{s}_{t-1}$ is the decoder's previous hidden state and $\text{score}(\cdot)$ measures compatibility between the decoder state and each encoder state.

<img src="https://raw.githubusercontent.com/farrelrassya/Transformers-in-action/main/folder-picture/01.%20chapter%2001/figure%201.2.png" width="700">


<img src="https://raw.githubusercontent.com/farrelrassya/Transformers-in-action/main/folder-picture/01.%20chapter%2001/figure%201.3.png" width="260">

In our English-to-French example, when the decoder generates "parle" (the French translation of "speak"), the attention weights $\alpha_{t,i}$ should assign the highest value to the encoder state corresponding to "speak," with meaningful but smaller weights on "I" and "don't" (which determine conjugation and negation). The word "French" would receive a lower weight at this particular decoding step, since it primarily influences the translation of itself rather than the verb form. The figure above visualizes this through edge thickness -- thicker edges represent higher attention weights.

**Why this solves the bottleneck:** Notice that the context vector $\mathbf{c}_t$ is now *different at every decoding step*. The decoder no longer relies on a single compressed representation. For a 200-word input sentence, the decoder can reach back and attend to word 3 when generating output word 150 -- the information never gets "forgotten" because it is always directly accessible. This is the architectural insight that made attention so transformative.



### 1.1.4 The Power of Multihead Attention

Standard attention computes a single set of weights $\alpha_{t,i}$, meaning the model has one "perspective" on which source positions matter. But language is full of simultaneous relationships. When translating "I don't speak French," the decoder generating "parle" needs to attend to "speak" (lexical meaning), to "I" (subject agreement -- first person singular), and to "don't" (negation). A single attention distribution must compromise between these competing demands.

**Multihead attention** solves this by running $H$ independent attention computations in parallel, each with its own learned projection matrices:

$$\text{head}_h = \text{Attention}(\mathbf{Q}\mathbf{W}_h^Q, \; \mathbf{K}\mathbf{W}_h^K, \; \mathbf{V}\mathbf{W}_h^V)$$

$$\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H) \, \mathbf{W}^O$$

Each head learns to focus on a *different type of relationship*. In practice, researchers have observed that individual heads specialize: one head might track syntactic dependencies (subject-verb agreement), another might capture semantic roles, and yet another might handle positional proximity.

<img src="https://raw.githubusercontent.com/farrelrassya/Transformers-in-action/main/folder-picture/01.%20chapter%2001/figure%201.4.png" width="700">

The sentiment classification example "The movie was not bad" demonstrates this beautifully. Understanding that the overall sentiment is **positive** requires recognizing that "not" modifies "bad," inverting its polarity. A single attention head might focus on "bad" and conclude negative sentiment. But with multiple heads, one head can capture the "not"-"bad" negation interaction while another tracks the subject "movie" -- and the model combines these perspectives to correctly classify the sentiment as positive.

An LSTM processing this sentence left-to-right would encode "not" into its hidden state at position 4, then encounter "bad" at position 5. Whether the negation signal in $\mathbf{h}_4$ survives the nonlinear update to $\mathbf{h}_5$ depends entirely on the learned gate values -- it *can* work, but it is fragile. In a transformer, the "not"-"bad" relationship is captured directly through attention, regardless of the distance between them.

### The BLEU Benchmark Result

The original transformer achieved a **BLEU score of 41** on English-to-French translation after only **3.5 days of training**. The **BLEU** (BiLingual Evaluation Understudy) metric measures how closely a machine translation matches human reference translations, computed as a modified precision over $n$-grams (typically $n = 1$ to $4$) with a brevity penalty:

$$\text{BLEU} = \text{BP} \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

where $p_n$ is the modified $n$-gram precision and $\text{BP}$ penalizes translations shorter than the reference. A BLEU score of 41 is exceptionally strong -- scores above 30 generally indicate fluent, understandable translations. What makes this result remarkable is not just the score itself, but the training efficiency: prior state-of-the-art LSTM-based systems required *weeks* of training to reach comparable performance. The parallelism enabled by attention -- processing all positions simultaneously rather than sequentially -- translates directly into wall-clock speedups.

This efficiency advantage compounded rapidly. Within just five years of the original paper (2017-2022), the community progressed from the base transformer to GPT, BERT, T5, GPT-3, and eventually ChatGPT -- a pace of innovation that would have been impossible under the sequential training constraints of RNN architectures.


### 1.2-1.3 Using Transformers in Practice: Pretrained Models and When to Apply Them

From a practitioner's perspective, the most important concept in these sections is **pretraining and fine-tuning** -- the two-stage paradigm that makes transformers practical. A **pretrained model** has already been trained on massive, general-purpose text corpora (billions of tokens from books, web pages, and other sources), learning broad linguistic knowledge: grammar, facts, reasoning patterns, and semantic relationships. When you fine-tune such a model on your specific task -- say, sentiment analysis with a labeled dataset of a few thousand examples -- you are leveraging all that prior knowledge rather than learning from scratch.

This matters enormously for cost and feasibility. Training a transformer from scratch on a language modeling objective can require thousands of GPU-hours and millions of dollars at LLM scale. Fine-tuning a pretrained model for sentiment classification might take **a few hours on a single GPU** -- a difference of several orders of magnitude.

The **Hugging Face Transformers library** has become the de facto standard for accessing pretrained models. Its practical value comes from three things: a unified API across model architectures (BERT, GPT-2, T5, etc.), thousands of community-contributed pretrained checkpoints, and tight integration with PyTorch and TensorFlow. For most NLP tasks -- classification, named entity recognition, summarization, translation, question answering -- the workflow is: (1) select a pretrained model appropriate for your task, (2) add a task-specific output head, (3) fine-tune on your labeled data, and (4) evaluate.

**When to choose transformers over simpler approaches:** Transformers are not always the right tool. For tabular data with well-engineered features, gradient-boosted trees (XGBoost, LightGBM) often outperform transformers with far less compute. For small text datasets (under a few hundred examples), a TF-IDF + logistic regression baseline can be surprisingly competitive and far more interpretable. Transformers shine when you have sequential or structural data with complex dependencies, when pretrained knowledge provides meaningful transfer, and when you have sufficient compute for inference. The decision should always be driven by the **accuracy-cost-interpretability tradeoff** for your specific use case, not by architectural novelty.

### Zero-Shot and Few-Shot Learning: Why Scale Unlocks Generalization

Two capabilities that emerge as transformers scale deserve careful attention: **zero-shot learning** and **few-shot learning**. These are not separate algorithms or training procedures -- they are emergent behaviors that arise from pretraining on sufficiently large and diverse corpora.

**Zero-shot learning** means the model performs a task it was never explicitly trained for, guided only by a natural language instruction. For example, a model pretrained on general text can be prompted with "Classify the following review as positive or negative: 'The food was bland and overpriced'" -- and produce the correct label, despite never having seen a sentiment classification objective during training. The model succeeds because its pretraining corpus contained enough examples of evaluative language, opinion expressions, and category labels that it implicitly learned the *structure* of classification tasks.

**Few-shot learning** provides the model with a handful of labeled examples (typically 1-5) directly in the prompt, allowing it to infer the task pattern from context:

$$\underbrace{\text{"Great film" → Positive, "Terrible acting" → Negative}}_{\text{few-shot examples}} \;,\; \text{"Loved every minute" → } \;?$$

The model leverages its pretrained knowledge to generalize from these sparse examples without updating any parameters -- no gradient descent, no backpropagation. This is pure **in-context learning**, and it becomes increasingly reliable as model size grows.

From a production standpoint, zero-shot and few-shot capabilities are transformative because they eliminate the traditional ML pipeline bottleneck: **labeled data collection**. In domains where labeling is expensive (legal document review, medical record classification), the ability to achieve reasonable performance with zero or a handful of examples dramatically reduces time-to-deployment. However, the accuracy gap between few-shot prompting and full fine-tuning remains significant for most tasks -- few-shot is best understood as a rapid prototyping tool, not a replacement for supervised training when labeled data is available.



### The Limitations of Scale

The progression from the original transformer ($\sim$$65$ million parameters) to early BERT ($110$ million) to GPT-3 ($175$ billion) and beyond follows an aggressive scaling trajectory. But **scale is not a free lunch**. Several concrete limitations emerge as models grow:

**Computational cost** scales superlinearly with parameter count. A model with $N$ parameters requires $O(N)$ memory just to store weights in FP32 ($N \times 4$ bytes), plus optimizer states during training (Adam requires $3\times$ the parameter memory for weights, momentum, and variance). A $175$ billion parameter model needs roughly $175 \times 10^9 \times 4 \approx 700$ GB just for weights -- far exceeding any single GPU's memory, requiring sophisticated model parallelism across dozens of accelerators.

**Domain specificity** is a subtler problem. A model pretrained on general web text has broad but shallow knowledge of specialized domains like healthcare, law, or finance. Medical terminology, regulatory language, and domain-specific reasoning patterns are underrepresented in general corpora. This means a billion-parameter general model may underperform a much smaller model fine-tuned on domain-specific data -- a critical insight for practitioners choosing architectures.

**Inference latency** matters in production. For real-time applications (chatbots, autocomplete, content moderation), the time to generate a response must stay under a few hundred milliseconds. Larger models produce higher quality outputs but at proportionally higher latency. The **accuracy-latency tradeoff** is one of the most important engineering decisions in deploying transformer-based systems, and it often favors smaller, distilled models over the largest available option.



## 1.4 From Transformer to LLM: The Lasting Blueprint

The key insight of this section is architectural: despite the enormous variety of models developed since 2017 -- BERT, GPT-2, GPT-3, T5, PaLM, LLaMA, Claude, and hundreds more -- **all of them are built on the same core blueprint** from *"Attention Is All You Need."* The differences between them lie in three dimensions:

**Architecture variant.** The original transformer uses a full **encoder-decoder** structure, ideal for sequence-to-sequence tasks like translation. Subsequent models diverged into two families: **encoder-only** models (BERT and its variants) that excel at understanding tasks (classification, NER, question answering), and **decoder-only** models (the GPT family) optimized for generation. The choice of variant determines what tasks the model naturally handles:

| Variant | Examples | Strength | Typical Tasks |
|---|---|---|---|
| Encoder-only | BERT, RoBERTa, ELECTRA | Bidirectional context | Classification, NER, extractive QA |
| Decoder-only | GPT-2/3/4, LLaMA, Claude | Autoregressive generation | Text completion, dialogue, code |
| Encoder-decoder | T5, BART, mBART | Seq-to-seq mapping | Translation, summarization, generative QA |

**Training strategy.** Modern LLMs combine multiple training phases: (1) **unsupervised pretraining** on raw text (next-token prediction or masked language modeling), (2) **supervised fine-tuning (SFT)** on curated instruction-response pairs, and (3) **reinforcement learning from human feedback (RLHF)** to align model outputs with human preferences. This three-stage pipeline -- pretrain, SFT, RLHF -- is the standard recipe behind models like ChatGPT and Claude, and each stage addresses a different aspect of model quality: linguistic competence, task-following ability, and alignment with human values, respectively.

**Attention refinements.** While the core scaled dot-product attention remains:

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\mathbf{V}$$

numerous refinements have been introduced to handle longer sequences and improve efficiency. **Sparse attention** (as in Longformer) reduces the $O(n^2)$ complexity by restricting which positions can attend to each other. **Rotary positional embeddings (RoPE)** encode position information directly into the attention computation, improving length generalization. **Grouped-query attention (GQA)** reduces memory cost during inference by sharing key-value heads across multiple query heads. All of these are modifications *within* the transformer blueprint, not departures from it.

The strategic takeaway: **understanding the original transformer architecture gives you transferable knowledge across every major LLM.** The components we will study in depth in the coming chapters -- embeddings, positional encodings, self-attention, feed-forward layers, residual connections, layer normalization -- appear in every modern model. Mastering them once means you can read any new architecture paper and immediately understand what has changed and what remains the same.



## Chapter 1 Summary: Key Takeaways

This chapter established *why* transformers exist and *what problem they solve*. The progression we traced is worth internalizing as a mental timeline:

**RNNs/LSTMs** processed sequences token-by-token, maintaining a compressed hidden state. This introduced two critical limitations: the **information bottleneck** (fixed-size context vector) and **sequential computation** ($O(n)$ non-parallelizable steps). These limitations made training slow and long-range dependency modeling unreliable.

**The attention mechanism** removed the information bottleneck by letting the decoder access all encoder states through learned, dynamic weights. This was initially added *on top of* RNNs (Bahdanau attention, 2014), but the 2017 transformer paper showed that attention alone -- without any recurrence -- was sufficient and superior.

**Multihead attention** extended this by running multiple independent attention computations in parallel, allowing the model to capture different types of relationships (syntactic, semantic, positional) simultaneously. The result: a BLEU score of **41** on English-to-French translation in just **3.5 days** of training.

**Pretraining and fine-tuning** made transformers practical for the broader community. Libraries like Hugging Face Transformers democratized access to models that would cost millions to train from scratch, enabling fine-tuning on task-specific data in hours rather than weeks.

**Zero-shot and few-shot learning** emerged as models scaled, allowing task performance without task-specific training data -- a capability with enormous practical implications for rapid prototyping and low-resource domains.

Finally, the **transformer-to-LLM pipeline** (pretraining, SFT, RLHF) represents the modern blueprint for building aligned, capable language models. Every model we will study in subsequent chapters is a variation on the architecture introduced here. The foundation is now set -- in Chapter 2, we will open the transformer and examine each component in mathematical detail.